### 1. Mã hóa và giải mã sử dụng thuật toán AES 256 (CBC Mode)

In [11]:
import subprocess
import os

def openssl_aes_256_encrypt(plaintext: bytes, passphrase: str) -> bytes:
    """Mã hóa AES-256-CBC bằng OpenSSL CLI với PBKDF2"""
    cmd = ['openssl', 'enc', '-aes-256-cbc', '-pbkdf2', '-pass', f'pass:{passphrase}']
    process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    stdout, stderr = process.communicate(input=plaintext)
    if process.returncode != 0:
        raise RuntimeError(f"OpenSSL Error: {stderr.decode('utf-8')}")
    return stdout

def openssl_aes_256_decrypt(ciphertext: bytes, passphrase: str) -> bytes:
    """Giải mã AES-256-CBC bằng OpenSSL CLI với PBKDF2"""
    cmd = ['openssl', 'enc', '-aes-256-cbc', '-d', '-pbkdf2', '-pass', f'pass:{passphrase}']
    process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    stdout, stderr = process.communicate(input=ciphertext)
    if process.returncode != 0:
        raise RuntimeError(f"OpenSSL Error: {stderr.decode('utf-8')}")
    return stdout

# --- Demo ---
msg = b"Day la lieu mat can ma hoa bang AES 256"
password = "MatKhau123"

cipher = openssl_aes_256_encrypt(msg, password)
decrypted = openssl_aes_256_decrypt(cipher, password)

print("[OpenSSL AES-256] Ciphertext (Hex):", cipher.hex())
print("[OpenSSL AES-256] Decrypted Message:", decrypted.decode('utf-8'))

[OpenSSL AES-256] Ciphertext (Hex): 53616c7465645f5fb6b5e50db37d184f4005e67f6161c40985d5b89da46b61d6dfee7fe0bd38275fa3a5e53f90ed9ec1ecdb2c0ed41b58516510364cf01a6494
[OpenSSL AES-256] Decrypted Message: Day la lieu mat can ma hoa bang AES 256


### 2. Mã hóa và giải mã phối hợp Elliptic Curve (ECDSA/ECDH SECP256R1 + AES)

In [12]:
import subprocess
import os

# 1. Tạo private key cho Sender & Receiver
subprocess.run(['openssl', 'ecparam', '-name', 'prime256v1', '-genkey', '-noout', '-out', 'sender_priv.pem'])
subprocess.run(['openssl', 'ecparam', '-name', 'prime256v1', '-genkey', '-noout', '-out', 'receiver_priv.pem'])

# 2. Trích xuất public key tương ứng
subprocess.run(['openssl', 'ec', '-in', 'sender_priv.pem', '-pubout', '-out', 'sender_pub.pem'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['openssl', 'ec', '-in', 'receiver_priv.pem', '-pubout', '-out', 'receiver_pub.pem'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 3. Thực hiện trao đổi khóa ECDH để tạo khóa dùng chung (Shared Secret)
# Người gửi tính toán shared secret bằng private key của mình và public key người nhận
proc_sender = subprocess.run(['openssl', 'pkeyutl', '-derive', '-inkey', 'sender_priv.pem', '-peerkey', 'receiver_pub.pem', '-out', 'shared_secret.bin'])

# Đọc shared secret được sinh ra làm khóa đối xứng AES-256
with open('shared_secret.bin', 'rb') as f:
    shared_key = f.read()

# Mã hóa thông điệp bằng khóa vừa thỏa thuận
msg_ec = b"Thong diep trao doi an toan qua ECDH-OpenSSL"
cipher_ec = openssl_aes_256_encrypt(msg_ec, shared_key.hex())
decrypt_ec = openssl_aes_256_decrypt(cipher_ec, shared_key.hex())

print("[OpenSSL ECDH] Shared Secret (Hex):", shared_key.hex())
print("[OpenSSL ECDH-AES] Decrypted:", decrypt_ec.decode('utf-8'))

[OpenSSL ECDH] Shared Secret (Hex): ab297e6b40059f8b47ae6c7031f0780f67d3d77bddf3fd493dcafaa77f6fa81d
[OpenSSL ECDH-AES] Decrypted: Thong diep trao doi an toan qua ECDH-OpenSSL


### 3. Ký và xác thực chữ ký sử dụng ECDSA

In [13]:
import subprocess

document_path = 'doc_to_sign.txt'
with open(document_path, 'wb') as f:
    f.write(b"Van ban kiem tra chu ky so ECDSA bang OpenSSL CLI.")

# 1. Ký số văn bản sử dụng khóa bí mật của người gửi (sender_priv.pem)
# Sử dụng thuật toán băm SHA256 kèm ECDSA
sig_path = 'signature.bin'
subprocess.run([
    'openssl', 'dgst', '-sha256',
    '-sign', 'sender_priv.pem',
    '-out', sig_path,
    document_path
])

with open(sig_path, 'rb') as f:
    sig_bytes = f.read()
print("[OpenSSL ECDSA] Signature (Hex):", sig_bytes.hex())

# 2. Xác thực chữ ký bằng khóa công khai của người gửi (sender_pub.pem)
verify_result = subprocess.run([
    'openssl', 'dgst', '-sha256',
    '-verify', 'sender_pub.pem',
    '-signature', sig_path,
    document_path
], capture_output=True, text=True)

print("[OpenSSL ECDSA] Verification Output:", verify_result.stdout.strip() or verify_result.stderr.strip())

[OpenSSL ECDSA] Signature (Hex): 30460221009fc9056057b37473879a2cbc18787d38fc5cca52df60bd7e750bd9d92daee665022100c6188901bd14625a1938f15188b74d9cc70f6827124186ec87a463e9a9f221f7
[OpenSSL ECDSA] Verification Output: Verified OK


In [14]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding, hashes
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.serialization import load_pem_public_key
from cryptography.hazmat.backends import default_backend
import hashlib

# 1. GIẢI MÃ KẾT QUẢ AES-256-CBC TỪ OPENSSL
print("=== 1. Giải mã AES-256-CBC (OpenSSL) bằng Python ===")

def decrypt_openssl_aes_256(ciphertext_with_salt: bytes, password: str) -> bytes:
    if not ciphertext_with_salt.startswith(b'Salted__'):
        raise ValueError("Không tìm thấy Salt định dạng OpenSSL!")

    salt = ciphertext_with_salt[8:16]
    actual_ciphertext = ciphertext_with_salt[16:]

    from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

    kdf = PBKDF2HMAC(
        algorithm=hashes.SHA256(),
        length=48,
        salt=salt,
        iterations=10000,
        backend=default_backend()
    )
    key_iv = kdf.derive(password.encode('utf-8'))
    key = key_iv[:32]
    iv = key_iv[32:]

    # Giải mã AES-CBC
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    decryptor = cipher.decryptor()
    padded_plaintext = decryptor.update(actual_ciphertext) + decryptor.finalize()

    # Gỡ bỏ PKCS7 Padding
    unpadder = padding.PKCS7(algorithms.AES.block_size).unpadder()
    plaintext = unpadder.update(padded_plaintext) + unpadder.finalize()
    return plaintext

try:
    decrypted_msg = decrypt_openssl_aes_256(cipher, password)
    print("[-] Giải mã thành công:", decrypted_msg.decode('utf-8'))
except Exception as e:
    print("[!] Lỗi giải mã AES:", e)


# 2. GIẢI MÃ KẾT QUẢ ECDH + AES TỪ OPENSSL
print("\n=== 2. Giải mã ECDH-AES (OpenSSL Shared Secret) bằng Python ===")
try:
    with open('shared_secret.bin', 'rb') as f:
        shared_key_from_openssl = f.read()

    decrypted_ec_msg = decrypt_openssl_aes_256(cipher_ec, shared_key_from_openssl.hex())
    print("[-] Giải mã ECDH-AES thành công:", decrypted_ec_msg.decode('utf-8'))
except Exception as e:
    print("[!] Lỗi giải mã ECDH-AES:", e)


# 3. XÁC THỰC CHỮ KÝ ECDSA TẠO BỞI OPENSSL
print("\n=== 3. Xác thực chữ ký ECDSA (OpenSSL) bằng Python ===")
try:
    with open('sender_pub.pem', 'rb') as f:
        public_key_bytes = f.read()

    sender_public_key = load_pem_public_key(public_key_bytes, backend=default_backend())

    with open('doc_to_sign.txt', 'rb') as f:
        original_document = f.read()

    with open('signature.bin', 'rb') as f:
        signature_from_openssl = f.read()

    sender_public_key.verify(
        signature_from_openssl,
        original_document,
        ec.ECDSA(hashes.SHA256())
    )
    print("[-] Xác thực chữ ký thành công: Chữ ký HỢP LỆ (Verified OK)!")
except Exception as e:
    print("[!] Xác thực chữ ký thất bại:", e)

=== 1. Giải mã AES-256-CBC (OpenSSL) bằng Python ===
[-] Giải mã thành công: Day la lieu mat can ma hoa bang AES 256

=== 2. Giải mã ECDH-AES (OpenSSL Shared Secret) bằng Python ===
[-] Giải mã ECDH-AES thành công: Thong diep trao doi an toan qua ECDH-OpenSSL

=== 3. Xác thực chữ ký ECDSA (OpenSSL) bằng Python ===
[-] Xác thực chữ ký thành công: Chữ ký HỢP LỆ (Verified OK)!
